# Building a Dynamic Programming Aligner using Python and Numpy

In this practical, you will implement a global sequence aligner using dynamic programming.

We will work with two protein sequences and compute an alignment by:

- defining a scoring scheme;

- constructing a matrix of scores;

- filling the matrix using a recurrence;

- reconstructing one optimal alignment via traceback.


We will build a matrix where each cell `(i, j)` stores the best score for aligning the first `i` residues of `seq1` with the first `j` residues of `seq2`

The full alignment is obtained by filling this matrix once and then tracing back through it.

A global alignment forces both sequences to be aligned end-to-end, allowing gaps:

`HEAGAWGHE-E`

`---PAW-HEAE`

Use this when sequences are of similar length and expected to be homologous across their full length.

**The implementation should be modular. Write small functions and test them as you go.**

**This will make it easier to later modify the code, for example to change the scoring scheme or implement local alignment.**

Let's initialise our sequences variables first. For example, let's use

`HEAGAWGHEE`

and

`PAWHEAE`

, store them in `seq1` and `seq2` and show them to screen using an f-string.


## Create the matrix

We will store alignment scores in a matrix.

The matrix has:

- `len(seq1) + 1` rows  

- `len(seq2) + 1` columns  

The extra row and column will be used later.

---

Write a function:

`make_matrix(n_rows, n_cols)`

that returns a matrix filled with zeros.

You can implement this either:

- using a list of lists  
- or using NumPy (e.g. `np.zeros`)

---

Test your function by creating a small matrix (e.g. 3 × 4) and printing it.

The general idea behind this is:

```
function make_matrix(n_rows, n_cols):

    create empty matrix

    for each row:
        create empty row

        for each column:
            add 0 to the row

        add row to matrix

    return matrix
```

Try to implement it as a function in Python

In [ ]:
def make_matrix(n_rows, n_cols):

    # create an empty list that will contain all rows


    # repeat once for each row
    for i in range(n_rows):

        # create a new empty row


        # fill this row with zeros
        for j in range(n_cols):
            .append(0)

        # add the completed row to the matrix
        .append(row)

    # return the full matrix
    return

What if we want to use Numpy for this? Feel free to continue in Numpy or basic Python.

In [ ]:
import numpy as np

def make_matrix(n_rows, n_cols):

    # shape of the matrix: (rows, columns)
    shape = (n_rows, n_cols)

    # Initialise a matrix of zeros with that shape. What's the function we need to use instead of XXX?
    matrix = np.XXX(shape, dtype=int)

    return matrix

## Create the score matrix for the sequences

We now use `make_matrix()` to create the matrix for our sequences.

The matrix should have:

- `len(seq1) + 1` rows  
- `len(seq2) + 1` columns  

The extra row and column are included so that we can index positions starting from 0.

---

### Task

1. Compute the number of rows and columns:
   - `n_rows = len(seq1) + 1`
   - `n_cols = len(seq2) + 1`
2. Create the matrix using `make_matrix(n_rows, n_cols)`.
3. Store it in a variable called `score_matrix`.
4. Print the matrix.

---

### Check

- What is the shape of the matrix?
- How does it compare to the lengths of `seq1` and `seq2`?

Verify your answer by printing `n_rows` and `n_cols`.

## Scoring residue pairs

We now define how to score the comparison between two residues.

For this first version, we only consider:

- match: `+1`
- mismatch: `-1`

We are not considering gaps yet.

---

### Task

Write a function:

`residue_score(a, b)`

that:

- returns `+1` if the two residues are the same
- returns `-1` otherwise

---

### Check

Test your function on a few examples:

- one where the residues are the same
- one where the residues are different

Print the results.

In [6]:
#def residue_score

print(residue_score("A", "A"))
print(residue_score("A", "G"))
print(residue_score("H", "H"))
print(residue_score("H", "P"))

## Filling the matrix: diagonal moves only

We now start filling the score matrix.

For this first version, we only allow **diagonal moves**.

We are still not considering gaps.

---

### What does a diagonal move mean?

A diagonal move goes from `(i-1, j-1)` to `(i, j)`.

This means we align:

- `seq1[i-1]`
- `seq2[j-1]`

For example:

- cell `(1, 1)` compares `seq1[0]` with `seq2[0]`
- cell `(2, 2)` compares `seq1[1]` with `seq2[1]`

The value of the new cell is:

**previous diagonal value + residue score**

In Python, this is:

`score_matrix[i][j] = score_matrix[i-1][j-1] + residue_score(seq1[i-1], seq2[j-1])`

---

### Task

Write a function:

`fill_diagonal(matrix, seq1, seq2)`

that fills the matrix using diagonal moves only.

For each internal cell `(i, j)`:

1. get the value from `(i-1, j-1)`
2. compare `seq1[i-1]` with `seq2[j-1]`
3. add the residue score to the previous diagonal value
4. store the result in `(i, j)`

Return the updated matrix.

---

### Check

After filling the matrix:

- which cells contain non-zero values?
- what limitation does this create?

## Why diagonal-only filling is not enough

The matrix has now been filled using diagonal moves only.

This means each cell only uses the value from the previous diagonal cell.

In other words, the algorithm always moves through both sequences at the same time.

---

### What this implies

Using our sequences:

`HEAGAWGHEE`

`PAWHEAE`

diagonal-only filling compares positions directly:

- `H` with `P`
- `E` with `A`
- `A` with `W`
- `G` with `H`
- `A` with `E`
- `W` with `A`
- `G` with `E`

This does not allow one sequence to shift relative to the other.

---

### Check

Look at the matrix you printed.

What kind of alignment is impossible if we only allow diagonal moves?

Write a short answer.

## Introducing gaps

To allow one sequence to shift relative to the other, we need gaps.

A gap is represented by `-`.

For example:

`HEAGAWGHE-E`

`---PAW-HEAE`

Gaps allow the alignment to skip a position in one sequence while continuing through the other.

---

### Gap penalty

We will use:

- gap: `-2`

This means that aligning a residue with a gap reduces the score by 2.

---

### Task

Define a variable called:

`gap_penalty`

and set it to `-2`.

## Filling one cell with gaps

We now update the scoring to include gaps.

Before writing the full function, we compute one cell by hand in code.

---

### Cell (1, 1)

Cell `(1, 1)` compares:

- `seq1[0]`
- `seq2[0]`

For this cell, we consider three possible moves:

---

### 1. Diagonal

From `(0, 0)`

Align:

`seq1[0]` with `seq2[0]`

Score:

`score_matrix[0][0] + residue_score(seq1[0], seq2[0])`

---

### 2. Up

From `(0, 1)`

Align:

`seq1[0]` with a gap

Score:

`score_matrix[0][1] + gap_penalty`

---

### 3. Left

From `(1, 0)`

Align:

a gap with `seq2[0]`

Score:

`score_matrix[1][0] + gap_penalty`

---

The value of the cell is the maximum of these three.

---

### Task

For cell `(1, 1)`:

1. print the two residues being compared
2. compute `diagonal`
3. compute `up`
4. compute `left`
5. print all three values
6. store the maximum in `score_matrix[1][1]`
7. print the matrix

---

### Check

Which of the three moves gives the highest score?

Why?

In case something went wrong at some point, please pick up the tutorial from here.

## Checkpoint: reset and initialise the matrix

Run this cell if you want to restart from a clean slate.

This cell defines:

- the sequences
- the scoring function
- the matrix creation function
- the matrix initialisation function
- a fresh score matrix ready for global alignment

---

The matrix is initialised so that:

- the first column contains cumulative gap penalties
- the first row contains cumulative gap penalties

This corresponds to aligning prefixes of each sequence with gaps.

After this step, the matrix is ready to be filled using the full dynamic programming algorithm.

In [1]:
import numpy as np

# Sequences
seq1 = "HEAGAWGHEE"
seq2 = "PAWHEAE"

# Gap penalty
gap_penalty = -2


# Scoring function
def residue_score(a, b):

    if a == "-":
        return gap_penalty
    elif b == "-":
        return gap_penalty
    elif a == b:
        return 1
    else:
        return -1


# Matrix creation
def make_matrix(n_rows, n_cols):

    return np.zeros((n_rows, n_cols), dtype=int)


# Matrix initialisation
def initialise_matrix(matrix, gap_penalty):

    n_rows, n_cols = matrix.shape

    # first column
    for i in range(n_rows):
        matrix[i, 0] = i * gap_penalty

    # first row
    for j in range(n_cols):
        matrix[0, j] = j * gap_penalty

    return matrix


# Matrix printing
def print_matrix(matrix, seq1, seq2):

    n_rows, n_cols = matrix.shape

    print("     ", end="")
    print("  ".join(["-"] + list(seq2)))

    for i in range(n_rows):

        if i == 0:
            row_label = "-"
        else:
            row_label = seq1[i - 1]

        print(row_label, end="  ")

        for j in range(n_cols):
            print(matrix[i, j], end="  ")

        print()


# Create and initialise matrix
n_rows = len(seq1) + 1
n_cols = len(seq2) + 1

score_matrix = make_matrix(n_rows, n_cols)
score_matrix = initialise_matrix(score_matrix, gap_penalty)

# Print
print_matrix(score_matrix, seq1, seq2)

     -  P  A  W  H  E  A  E
-  0  -2  -4  -6  -8  -10  -12  -14  
H  -2  0  0  0  0  0  0  0  
E  -4  0  0  0  0  0  0  0  
A  -6  0  0  0  0  0  0  0  
G  -8  0  0  0  0  0  0  0  
A  -10  0  0  0  0  0  0  0  
W  -12  0  0  0  0  0  0  0  
G  -14  0  0  0  0  0  0  0  
H  -16  0  0  0  0  0  0  0  
E  -18  0  0  0  0  0  0  0  
E  -20  0  0  0  0  0  0  0  


## Filling the matrix (global alignment)

We now fill the entire matrix using the three possible moves.

You have already computed one cell `(1, 1)` manually.

Now you will apply the same logic to every cell in the matrix.

---

### For each cell `(i, j)`

We consider three possible ways to reach it:

---

#### 1. Diagonal

From `(i-1, j-1)`

Align:

- `seq1[i-1]`
- `seq2[j-1]`

Score:

`score_matrix[i-1][j-1] + residue_score(seq1[i-1], seq2[j-1])`

---

#### 2. Up

From `(i-1, j)`

Align:

- `seq1[i-1]`
- gap (`-`)

Score:

`score_matrix[i-1][j] + gap_penalty`

---

#### 3. Left

From `(i, j-1)`

Align:

- gap (`-`)
- `seq2[j-1]`

Score:

`score_matrix[i][j-1] + gap_penalty`

---

### Final value

The value of the cell is the maximum of the three scores:

- diagonal
- up
- left

---

### Task

Write a function:

`fill_matrix(matrix, seq1, seq2, gap_penalty)`

that:

- loops over all cells starting from `(1, 1)`
- computes the three scores for each cell
- stores the maximum value in the matrix
- returns the filled matrix

Then print the matrix.

---

### Check

After filling the matrix:

- where is the final alignment score located?
- how does it relate to the lengths of the sequences?

In [ ]:
def fill_matrix(matrix, seq1, seq2, gap_penalty):

    n_rows, n_cols = matrix.shape

    # loop over all cells, starting from (1, 1)
    for i in range(1, n_rows):
        for j in range(1, n_cols):

            # diagonal move: match or mismatch
            diagonal = matrix[i-1, j-1] + residue_score(seq1[i-1], seq2[j-1])

            # up move: gap in seq2
            up = matrix[i-1, j] + gap_penalty

            # left move: gap in seq1
            left = matrix[i, j-1] + gap_penalty

            # choose the best score
            matrix[i, j] = max(diagonal, up, left)

    return matrix


# apply function
score_matrix = fill_matrix(score_matrix, seq1, seq2, gap_penalty)

# print matrix
print_matrix(score_matrix, seq1, seq2)

## Traceback: recovering the alignment

The filled matrix gives us the best alignment score.

The score alone is not enough: we also want to recover the alignment.

To do this, we trace back through the matrix.

---

### Starting point

For global alignment, traceback starts from the bottom-right cell of the matrix.

This cell corresponds to aligning:

- all of `seq1`
- all of `seq2`

---

### At each step

We ask which previous cell could have produced the current score.

There are three possibilities:

---

#### 1. Diagonal

If the current score came from `(i-1, j-1)`, then align:

- `seq1[i-1]`
- `seq2[j-1]`

---

#### 2. Up

If the current score came from `(i-1, j)`, then align:

- `seq1[i-1]`
- gap (`-`)

---

#### 3. Left

If the current score came from `(i, j-1)`, then align:

- gap (`-`)
- `seq2[j-1]`

---

### Important

Traceback builds the alignment backwards.

This means we collect residues from the end of the alignment to the beginning.

At the end, we reverse both aligned sequences.

---

### Task

Write a function:

`traceback(matrix, seq1, seq2, gap_penalty)`

that returns two aligned sequences.

# An all-in-one cell to do dynamic programming

Congratulations, you made it to the end, or I hope so.

Don't be alarmed by the amount of code we covered. Most of the cells are there to trace back how we get to the final cell, an all-in-one that takes your sequences and aligns them. The previous cells should help you realise how the final functions are working.



In [2]:
import numpy as np

# Sequences
seq1 = "HEAGAWGHEE"
seq2 = "PAWHEAE"

# Scoring
match_score = 1
mismatch_score = -1
gap_penalty = -2


def residue_score(a, b):
    if a == b:
        return match_score
    else:
        return mismatch_score


def make_matrix(n_rows, n_cols):
    matrix = np.zeros((n_rows, n_cols), dtype=int)
    return matrix


def initialise_matrix(matrix, gap_penalty):
    n_rows, n_cols = matrix.shape

    for i in range(n_rows):
        matrix[i, 0] = i * gap_penalty

    for j in range(n_cols):
        matrix[0, j] = j * gap_penalty

    return matrix


def fill_matrix(matrix, seq1, seq2, gap_penalty):
    n_rows, n_cols = matrix.shape

    for i in range(1, n_rows):
        for j in range(1, n_cols):

            diagonal = matrix[i-1, j-1] + residue_score(seq1[i-1], seq2[j-1])
            up = matrix[i-1, j] + gap_penalty
            left = matrix[i, j-1] + gap_penalty

            matrix[i, j] = max(diagonal, up, left)

    return matrix


def traceback(matrix, seq1, seq2, gap_penalty):
    aligned_seq1 = []
    aligned_seq2 = []

    i = len(seq1)
    j = len(seq2)

    while i > 0 or j > 0:

        current_score = matrix[i, j]

        if i > 0 and j > 0:
            diagonal = matrix[i-1, j-1] + residue_score(seq1[i-1], seq2[j-1])
        else:
            diagonal = None

        if i > 0:
            up = matrix[i-1, j] + gap_penalty
        else:
            up = None

        if j > 0:
            left = matrix[i, j-1] + gap_penalty
        else:
            left = None

        if diagonal == current_score:
            aligned_seq1.append(seq1[i-1])
            aligned_seq2.append(seq2[j-1])
            i -= 1
            j -= 1

        elif up == current_score:
            aligned_seq1.append(seq1[i-1])
            aligned_seq2.append("-")
            i -= 1

        elif left == current_score:
            aligned_seq1.append("-")
            aligned_seq2.append(seq2[j-1])
            j -= 1

        else:
            raise ValueError("No valid traceback move found.")

    aligned_seq1.reverse()
    aligned_seq2.reverse()

    return "".join(aligned_seq1), "".join(aligned_seq2)


def print_matrix(matrix, seq1, seq2):
    n_rows, n_cols = matrix.shape

    print("     ", end="")
    print("  ".join(["-"] + list(seq2)))

    for i in range(n_rows):

        if i == 0:
            row_label = "-"
        else:
            row_label = seq1[i - 1]

        print(row_label, end="  ")

        for j in range(n_cols):
            print(matrix[i, j], end="  ")

        print()


def make_match_line(aligned_seq1, aligned_seq2):
    line = []

    for a, b in zip(aligned_seq1, aligned_seq2):
        if a == b:
            line.append("|")
        else:
            line.append(" ")

    return "".join(line)


# Run alignment
n_rows = len(seq1) + 1
n_cols = len(seq2) + 1

score_matrix = make_matrix(n_rows, n_cols)
score_matrix = initialise_matrix(score_matrix, gap_penalty)
score_matrix = fill_matrix(score_matrix, seq1, seq2, gap_penalty)

aligned_seq1, aligned_seq2 = traceback(score_matrix, seq1, seq2, gap_penalty)

# Output
print("Score matrix:")
print_matrix(score_matrix, seq1, seq2)

print()
print("Alignment:")
print(aligned_seq1)
print(make_match_line(aligned_seq1, aligned_seq2))
print(aligned_seq2)

print()
print("Alignment score:", score_matrix[-1, -1])

Score matrix:
     -  P  A  W  H  E  A  E
-  0  -2  -4  -6  -8  -10  -12  -14  
H  -2  -1  -3  -5  -5  -7  -9  -11  
E  -4  -3  -2  -4  -6  -4  -6  -8  
A  -6  -5  -2  -3  -5  -6  -3  -5  
G  -8  -7  -4  -3  -4  -6  -5  -4  
A  -10  -9  -6  -5  -4  -5  -5  -6  
W  -12  -11  -8  -5  -6  -5  -6  -6  
G  -14  -13  -10  -7  -6  -7  -6  -7  
H  -16  -15  -12  -9  -6  -7  -8  -7  
E  -18  -17  -14  -11  -8  -5  -7  -7  
E  -20  -19  -16  -13  -10  -7  -6  -6  

Alignment:
HEAGAWGHE-E
    || || |
---PAW-HEAE

Alignment score: -6


## Exercises

1. Change the gap penalty to `-1`. How does the alignment change?
2. Change the match score to `+2`. How does the score change?
3. Try `seq1 = "GATTACA"` and `seq2 = "GCATGCU"`.
4. What changes would be needed for local alignment?